# Grasp Detection with DepthAnythingV2 Small
## Cornell Grasping Dataset Evaluation

**Author:** Steyn Knollema (knollema@seas.upenn.edu)

This notebook implements the heuristic-based grasp detection pipeline using:
- **DepthAnythingV2 Small** for monocular depth estimation (replacing MiDaS)
- **Cornell Grasping Dataset** for evaluation
- **IoU-based accuracy metrics** comparing predictions to ground truth

### Pipeline Overview:
1. RGB Image → DepthAnythingV2 Small → Pseudo Depth
2. Edge Detection (Canny) + Depth Gradients (Sobel)
3. Saliency Fusion → Candidate Extraction → Filtering
4. Quality Scoring (Edge + Depth Gradient + CoG Proximity)
5. Compare with Cornell Ground Truth Rectangles


## 1. Installation and Imports


In [2]:
# Install required packages

!pip install -q torch torchvision opencv-python matplotlib numpy Pillow tqdm scipy transformers huggingface_hub opendatasets kaggle



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Polygon
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import os
import glob
import time
import json
import zipfile
import shutil
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Using device: cpu


## 2. Download Cornell Dataset from Kaggle

**Option A:** Use `opendatasets` (easiest - will prompt for credentials)  
**Option B:** Use Kaggle API (requires kaggle.json setup)  


In [4]:

DATASET_DIR = './cornell_dataset'
KAGGLE_DATASET = 'oneoneliu/cornell-grasp'

def download_cornell_kaggle_opendatasets():
    import opendatasets as od

    print("="*80)
    print("DOWNLOADING CORNELL GRASPING DATASET FROM KAGGLE")
    print("="*80)
    print("\nYou'll need your Kaggle credentials:")
    print("1. Go to https://www.kaggle.com/account")
    print("2. Scroll to 'API' section")
    print("3. Click 'Create New Token' to download kaggle.json")
    print("4. Enter username and key when prompted below\n")

    # Download dataset
    od.download(f"https://www.kaggle.com/datasets/{KAGGLE_DATASET}")

    # Move to expected location
    downloaded_dir = './cornell-grasp'
    if os.path.exists(downloaded_dir):
        if os.path.exists(DATASET_DIR):
            shutil.rmtree(DATASET_DIR)
        shutil.move(downloaded_dir, DATASET_DIR)

    return True

def download_cornell_kaggle_api():
    import kaggle

    print("Downloading via Kaggle API...")

    os.makedirs(DATASET_DIR, exist_ok=True)

    # Download and unzip
    kaggle.api.dataset_download_files(
        KAGGLE_DATASET,
        path=DATASET_DIR,
        unzip=True
    )

    return True

def setup_kaggle_credentials():
    print("="*80)
    print("KAGGLE CREDENTIALS SETUP")
    print("="*80)

    kaggle_dir = os.path.join(os.path.expanduser('~'), '.kaggle')
    kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')

    if os.path.exists(kaggle_json):
        return True

    print("\nNo kaggle.json found. Let's set it up!")
    print("\n1. Go to https://www.kaggle.com/account")
    print("2. Scroll down to 'API' section")
    print("3. Click 'Create New Token'")
    print("4. This downloads kaggle.json\n")

    # Get credentials from user
    username = input("Enter your Kaggle username: ").strip()
    key = input("Enter your Kaggle API key: ").strip()

    if not username or not key:
        return False

    os.makedirs(kaggle_dir, exist_ok=True)

    credentials = {"username": username, "key": key}
    with open(kaggle_json, 'w') as f:
        json.dump(credentials, f)

    try:
        os.chmod(kaggle_json, 0o600)
    except:
        pass

    return True

def check_dataset_exists():
    if not os.path.exists(DATASET_DIR):
        return False

    # Look for image files
    patterns = [
        os.path.join(DATASET_DIR, '**', 'pcd*r.png'),
        os.path.join(DATASET_DIR, '**', '*.png'),
    ]

    for pattern in patterns:
        files = glob.glob(pattern, recursive=True)
        if len(files) > 10:
            return True

    return False

print("Cornell Dataset Downloader Ready")
print("Run the next cell to download the dataset")

Cornell Dataset Downloader Ready
Run the next cell to download the dataset


In [5]:

if check_dataset_exists():
    print("Dataset already downloaded!")
else:
    print("Dataset not found. Starting download...\n")

    try:
        download_cornell_kaggle_opendatasets()
    except Exception as e:
        print(f"\nopendatasets failed: {e}")
        print("\nTrying Kaggle API method...\n")

        if setup_kaggle_credentials():
            try:
                download_cornell_kaggle_api()
            except Exception as e2:
                print(f"\nKaggle API also failed: {e2}")
                print("\n" + "="*80)
                print("MANUAL DOWNLOAD REQUIRED")
                print("="*80)
                print("\n1. Go to: https://www.kaggle.com/datasets/oneoneliu/cornell-grasp")
                print("2. Click 'Download' button")
                print("3. Extract the zip file")
                print(f"4. Move contents to: {os.path.abspath(DATASET_DIR)}")
                print("5. Re-run this cell to verify")

Dataset already downloaded!


In [6]:

EDGE_WEIGHT = 0.01
DEPTH_WEIGHT = 0.04
COG_WEIGHT = 0.95

# Verify weights sum to 1.0
total_weight = EDGE_WEIGHT + DEPTH_WEIGHT + COG_WEIGHT
assert abs(total_weight - 1.0) < 0.01

BOUNDARY_MARGIN = 50
DEPTH_MIN = 0.15
DEPTH_MAX = 0.90
TEXTURE_THRESHOLD = 5
TEXTURE_PATCH_SIZE = 20
MIN_NEIGHBORS = 3
NEIGHBOR_RADIUS = 40

IOU_THRESHOLD = 0.25
ANGLE_THRESHOLD = 30

SHOW_TOP_PERCENT = 10
COLORMAP = 'RdYlGn_r'

print("="*80)
print("CONFIGURATION PARAMETERS")
print("="*80)
print("NOTE: Main pipeline params are in the GraspDetector cell (cell 22)")
print("-"*80)
print(f"Legacy Weights: Edge={EDGE_WEIGHT*100:.1f}%, Depth={DEPTH_WEIGHT*100:.1f}%, CoG={COG_WEIGHT*100:.1f}%")
print(f"IoU Threshold:       {IOU_THRESHOLD}")
print(f"Angle Threshold:     {ANGLE_THRESHOLD}°")
print("="*80)


CONFIGURATION PARAMETERS
NOTE: Main pipeline params are in the GraspDetector cell (cell 22)
--------------------------------------------------------------------------------
Legacy Weights: Edge=1.0%, Depth=4.0%, CoG=95.0%
IoU Threshold:       0.25
Angle Threshold:     30°


## 4. DepthAnythingV2 Small Model Setup


In [8]:
class DepthAnythingV2:

    def __init__(self, model_size='small', device='cuda'):
        self.device = device
        self.model_size = model_size
        self.model = None
        self.processor = None
        self.load_time = 0

    def load_model(self):
        start_time = time.time()

        try:
            from transformers import AutoImageProcessor, AutoModelForDepthEstimation

            model_ids = {
                'small': 'depth-anything/Depth-Anything-V2-Small-hf',
                'base': 'depth-anything/Depth-Anything-V2-Base-hf',
                'large': 'depth-anything/Depth-Anything-V2-Large-hf'
            }

            model_id = model_ids.get(self.model_size, model_ids['small'])
            print(f"Loading {model_id}...")

            self.processor = AutoImageProcessor.from_pretrained(model_id)
            self.model = AutoModelForDepthEstimation.from_pretrained(model_id)
            self.model.to(self.device)
            self.model.eval()

            self.load_time = time.time() - start_time

        except Exception as e:
            print(f"Error loading model: {e}")
            print("Falling back to MiDaS...")
            self._load_midas_fallback()

    def _load_midas_fallback(self):
        start_time = time.time()

        self.model = torch.hub.load('intel-isl/MiDaS', 'DPT_Hybrid', pretrained=True)
        self.model.to(self.device)
        self.model.eval()

        midas_transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
        self.transform = midas_transforms.dpt_transform
        self.processor = None

        self.load_time = time.time() - start_time

    @torch.no_grad()
    def predict(self, image: np.ndarray):
        start_time = time.time()

        h, w = image.shape[:2]

        if self.processor is not None:
            pil_image = Image.fromarray(image)
            inputs = self.processor(images=pil_image, return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            outputs = self.model(**inputs)
            depth = outputs.predicted_depth

            depth = F.interpolate(
                depth.unsqueeze(1),
                size=(h, w),
                mode='bicubic',
                align_corners=False
            ).squeeze().cpu().numpy()
        else:
            # MiDaS fallback path
            input_batch = self.transform(image).to(self.device)
            prediction = self.model(input_batch)

            depth = F.interpolate(
                prediction.unsqueeze(1),
                size=(h, w),
                mode='bicubic',
                align_corners=False
            ).squeeze().cpu().numpy()

        # Normalize to [0, 1]
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

        inference_time = (time.time() - start_time) * 1000

        return depth.astype(np.float32), inference_time


In [9]:
print("Initializing DepthAnythingV2 Small...")
depth_model = DepthAnythingV2(model_size='small', device=str(device))
depth_model.load_model()

Initializing DepthAnythingV2 Small...
Loading depth-anything/Depth-Anything-V2-Small-hf...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## 5. Cornell Grasping Dataset Handler


In [17]:
class CornellGraspDataset:
    def __init__(self, data_dir: str = './cornell_dataset'):
        self.data_dir = data_dir
        self.samples = []
    
    def load_dataset(self, max_samples: Optional[int] = None):
        self.samples = []

        patterns = [
            os.path.join(self.data_dir, '**', 'pcd*r.png'),
            os.path.join(self.data_dir, '**', '*r.png'),
            os.path.join(self.data_dir, '*', 'pcd*r.png'),
        ]

        rgb_files = []
        for pattern in patterns:
            rgb_files.extend(glob.glob(pattern, recursive=True))
        rgb_files = list(set(rgb_files))

        print(f"Found {len(rgb_files)} RGB images")

        for rgb_path in sorted(rgb_files):
            if rgb_path.endswith('r.png'):
                base = rgb_path[:-5]
            else:
                continue

            # Find grasp file
            grasp_path = base + 'cpos.txt'
            if not os.path.exists(grasp_path):
                continue

            # Load grasps
            grasps = self._load_grasp_rectangles(grasp_path)
            if not grasps:
                continue

            # Depth is optional
            depth_path = base + 'd.tiff'
            if not os.path.exists(depth_path):
                depth_path = None

            self.samples.append({
                'rgb_path': rgb_path,
                'depth_path': depth_path,
                'grasps': grasps,
                'id': os.path.basename(base)
            })

            if max_samples and len(self.samples) >= max_samples:
                break

        return self.samples

    def _load_grasp_rectangles(self, filepath: str):
        grasps = []

        try:
            with open(filepath, 'r') as f:
                lines = f.readlines()

            for i in range(0, len(lines) - 3, 4):
                corners = []
                valid = True

                for j in range(4):
                    try:
                        parts = lines[i + j].strip().split()
                        if len(parts) >= 2:
                            x, y = float(parts[0]), float(parts[1])
                            if not (np.isnan(x) or np.isnan(y)):
                                corners.append([x, y])
                            else:
                                valid = False
                                break
                        else:
                            valid = False
                            break
                    except:
                        valid = False
                        break

                if valid and len(corners) == 4:
                    corners = np.array(corners)

                    # Calculate center
                    center = corners.mean(axis=0)

                    # Calculate angle
                    dx = corners[1, 0] - corners[0, 0]
                    dy = corners[1, 1] - corners[0, 1]
                    angle = np.arctan2(dy, dx)

                    # Calculate dimensions
                    width = np.linalg.norm(corners[1] - corners[0])
                    height = np.linalg.norm(corners[2] - corners[1])

                    grasps.append({
                        'corners': corners,
                        'center': center,
                        'angle': angle,
                        'angle_deg': np.degrees(angle),
                        'width': width,
                        'height': height
                    })
        except Exception as e:
            pass

        return grasps


In [18]:
# Load dataset
dataset = CornellGraspDataset(DATASET_DIR)
samples = dataset.load_dataset(max_samples=850)  # Limit for speed

if not samples:
    print(f"Expected location: {os.path.abspath(DATASET_DIR)}")

Found 896 RGB images


## 6. Heuristic Grasp Detection Pipeline


In [19]:

import cv2
import numpy as np


# CROPPING FUNCTIONS

def manual_crop(image, crop_bbox):
    """Manually crop image to specified region."""
    x, y, w, h = crop_bbox
    img_h, img_w = image.shape[:2]
    x = max(0, min(x, img_w - 1))
    y = max(0, min(y, img_h - 1))
    w = min(w, img_w - x)
    h = min(h, img_h - y)
    cropped = image[y:y+h, x:x+w].copy()
    return cropped, (x, y, w, h), True


def adjust_grasp_coordinates(grasps, crop_bbox):
    """Adjust ground truth grasp coordinates after cropping."""
    x_offset, y_offset, crop_w, crop_h = crop_bbox
    adjusted_grasps = []

    for grasp in grasps:
        center_x, center_y = grasp['center']
        new_center_x = center_x - x_offset
        new_center_y = center_y - y_offset

        if 0 <= new_center_x < crop_w and 0 <= new_center_y < crop_h:
            adjusted_corners = []
            for corner in grasp['corners']:
                adj_corner = (corner[0] - x_offset, corner[1] - y_offset)
                adjusted_corners.append(adj_corner)

            adjusted_grasp = {
                'center': (new_center_x, new_center_y),
                'angle': grasp['angle'],
                'angle_deg': grasp['angle_deg'],
                'width': grasp['width'],
                'height': grasp['height'],
                'corners': np.array(adjusted_corners)
            }
            adjusted_grasps.append(adjusted_grasp)

    return adjusted_grasps


# =============================================================================
# CONFIGURATION
# =============================================================================

# MANUAL CROP REGION
# Horizontal: 100 to 500 (x=100, width=400)
# Vertical: 150 to 450 (y=150, height=300)
MANUAL_CROP_BBOX = (100, 150, 400, 300)

ENABLE_MANUAL_CROP = True

print("=" * 80)
print("CORNELL EVALUATION WITH MANUAL CROP")
print("=" * 80)
print(f"Manual Crop Enabled: {ENABLE_MANUAL_CROP}")
x, y, w, h = MANUAL_CROP_BBOX
print(f"Crop Region:")
print(f"  Horizontal: {x} to {x+w} (width={w})")
print(f"  Vertical:   {y} to {y+h} (height={h})")
print(f"  Format: x={x}, y={y}, width={w}, height={h}")
print("=" * 80)

# =============================================================================
# STEP 1: CROP + DEPTH COMPUTATION
# =============================================================================

cached_data_manual = []
crop_stats = {
    'total': 0,
    'cropped': 0,
    'skipped_no_grasps': 0,
    'grasps_removed': 0,
    'grasps_kept': 0
}

print("\nStep 1: Manual cropping and depth computation...")
start_time = time.time()

for sample in tqdm(samples, desc="Manual crop + depth"):
    try:
        # Load RGB
        rgb = cv2.imread(sample["rgb_path"])
        if rgb is None:
            continue
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        crop_stats['total'] += 1
        grasps_to_use = sample["grasps"]
        crop_info = {'cropped': False}

        # MANUAL CROP
        if ENABLE_MANUAL_CROP:
            cropped_rgb, crop_bbox, success = manual_crop(rgb, MANUAL_CROP_BBOX)

            if success:
                # Adjust ground truth
                original_grasp_count = len(sample["grasps"])
                adjusted_grasps = adjust_grasp_coordinates(sample["grasps"], crop_bbox)

                grasps_removed = original_grasp_count - len(adjusted_grasps)
                crop_stats['grasps_removed'] += grasps_removed
                crop_stats['grasps_kept'] += len(adjusted_grasps)

                if len(adjusted_grasps) > 0:
                    rgb = cropped_rgb
                    grasps_to_use = adjusted_grasps
                    crop_stats['cropped'] += 1
                    crop_info = {
                        'cropped': True,
                        'crop_bbox': crop_bbox,
                        'grasps_removed': grasps_removed
                    }
                else:
                    # All grasps outside crop region - skip this sample
                    crop_stats['skipped_no_grasps'] += 1
                    continue

        # Compute depth
        depth, _ = depth_model.predict(rgb)

        cached_data_manual.append({
            "rgb": rgb,
            "depth": depth,
            "grasps": grasps_to_use,
            "crop_info": crop_info,
            "original_path": sample["rgb_path"]
        })

    except Exception as e:
        print(f"\nError: {sample.get('rgb_path', 'unknown')}: {e}")
        continue

depth_time = time.time() - start_time

print(f"\n✓ Processed {len(cached_data_manual)}/{len(samples)} samples")
if ENABLE_MANUAL_CROP:
    print(f"\n{'='*80}")
    print("MANUAL CROP STATISTICS")
    print(f"{'='*80}")
    print(f"Total samples:               {crop_stats['total']}")
    print(f"Successfully cropped:        {crop_stats['cropped']} ({crop_stats['cropped']/crop_stats['total']*100:.1f}%)")
    print(f"Skipped (no grasps in crop): {crop_stats['skipped_no_grasps']}")
    print(f"Samples with data:           {len(cached_data_manual)}")
    print(f"Grasps kept:                 {crop_stats['grasps_kept']}")
    print(f"Grasps removed (outside):    {crop_stats['grasps_removed']}")
    print(f"{'='*80}")

print(f"\nDepth computation time: {depth_time:.1f}s")

# =============================================================================
# STEP 2: GRASP DETECTION
# =============================================================================

print("\nStep 2: Running grasp detection on manually cropped images...")

detector = GraspDetector(
    BEST_PARAMS["w_edge"],
    BEST_PARAMS["w_depth"],
    BEST_PARAMS["w_cog"],
)

top1_successes = 0
top5_successes = 0
any_successes = 0
ious = []
angle_diffs = []
total = 0

start_time = time.time()

for data in tqdm(cached_data_manual, desc="Evaluating"):
    try:
        # Run grasp detection
        grasps, info = detector.process(
            rgb=data["rgb"],
            depth=data["depth"],
            n_grasps=BEST_PARAMS["num_grasps"],
            pct=BEST_PARAMS["depth_percentile"],
            mult=BEST_PARAMS["candidate_multiplier"],
            min_l=BEST_PARAMS["min_grasp_length"],
            max_l=BEST_PARAMS["max_grasp_length"],
            algo=BEST_PARAMS["ray_algorithm"],
            boost=BEST_PARAMS["cog_boost"],
            grad_src=BEST_PARAMS["gradient_source"]
        )

        # Convert to GraspCandidate
        predictions = [
            GraspCandidate(x=g.x, y=g.y, angle=g.angle, width=g.width, height=g.height)
            for g in grasps
        ]

        # Evaluate
        eval_result = evaluator.evaluate_image(predictions, data["grasps"])

        total += 1
        if eval_result.get("top1_success", False):
            top1_successes += 1
        if eval_result.get("top5_success", False):
            top5_successes += 1
        if eval_result.get("any_success", False):
            any_successes += 1

        ious.append(eval_result.get("avg_iou", 0.0))
        angle_diffs.append(eval_result.get("avg_angle_diff", 0.0))

        # Store for visualization
        data["eval_result"] = eval_result
        data["predictions"] = predictions
        data["detection_info"] = info

    except Exception as e:
        print(f"\nError: {e}")
        continue

eval_time = time.time() - start_time

# =============================================================================
# STEP 3: RESULTS
# =============================================================================

if total == 0:
    print("\n❌ No samples evaluated.")
else:
    top1_acc = top1_successes / total * 100.0
    top5_acc = top5_successes / total * 100.0
    any_acc = any_successes / total * 100.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    mean_angle = float(np.mean(angle_diffs)) if angle_diffs else 0.0

    print("\n" + "=" * 80)
    print("RESULTS WITH MANUAL CROP")
    print("=" * 80)
    print(f"Crop Region: Horizontal 100-500, Vertical 150-450")
    print(f"  (x=100, y=150, width=400, height=300)")
    print("-" * 80)
    print(f"Images evaluated          : {total}")
    print(f"Top-1 accuracy            : {top1_acc:.2f}%")
    print(f"Top-5 accuracy            : {top5_acc:.2f}%")
    print(f"Any-success accuracy      : {any_acc:.2f}%")
    print(f"Average IoU               : {mean_iou:.4f}")
    print(f"Average angle difference  : {mean_angle:.2f}°")
    print("-" * 80)
    print(f"Evaluation time           : {eval_time:.1f}s")
    print(f"Total time                : {depth_time + eval_time:.1f}s")
    print("=" * 80)

# Store for visualization
cached_data = cached_data_manual




# CONFIGURABLE PARAMETERS

W_EDGE = 0.001
W_DEPTH = 0.001
W_COG = 0.999

# Masking parameters
DEPTH_PERCENTILE = 30

# Ray casting parameters
RAY_ALGORITHM = "Direct Line with CoG Boost"
COG_BOOST_VALUE = 3.75
GRADIENT_SOURCE = "Contour Direction (80px avg)"

# Filtering parameters
MIN_GRASP_LENGTH = 1
MAX_GRASP_LENGTH = 1000

# Output parameters
NUM_OUTPUT_GRASPS = 1
CANDIDATE_MULTIPLIER =  100

print("="*80)
print("PIPELINE PARAMETERS (app_working.py method)")
print("="*80)
print(f"Weights: Edge={W_EDGE:.2f}, Depth={W_DEPTH:.2f}, CoG={W_COG:.2f}")
print(f"Depth Percentile: {DEPTH_PERCENTILE}")
print(f"Ray Algorithm: {RAY_ALGORITHM}")
print(f"Grasp Length Range: [{MIN_GRASP_LENGTH}, {MAX_GRASP_LENGTH}]")
print(f"Output Grasps: {NUM_OUTPUT_GRASPS}, Candidate Multiplier: {CANDIDATE_MULTIPLIER}")
print("="*80)

class Candidate:
    def __init__(self, x, y, angle, w, h=20.0, eq=0.0, dq=0.0, cq=0.0, comb=0.0, l=0.0):
        self.x = int(x)
        self.y = int(y)
        self.angle = angle
        self.width = float(w)
        self.height = h
        self.edge_quality = float(eq)
        self.depth_quality = float(dq)
        self.cog_quality = float(cq)
        self.combined_quality = float(comb)
        self.line_length = float(l)

    def get_corners(self):
        c = np.cos(self.angle)
        s = np.sin(self.angle)
        w, h = self.width, self.height

        # local corners
        pts = [(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]

        res = []
        for dx, dy in pts:
            px = self.x + dx * c - dy * s
            py = self.y + dx * s + dy * c
            res.append([px, py])

        return np.array(res)

class GraspDetector:
    def __init__(self, w_e, w_d, w_c):
        self.we = w_e
        self.wd = w_d
        self.wc = w_c

    def _ray_cast(self, mask, start, vec_x, vec_y, contour=None, mode="Depth Gradients", limit=500):
        h, w = mask.shape
        gx, gy = start

        # Determine direction (dx, dy)
        dx, dy = 1.0, 0.0 # defaults

        if mode == "Contour Direction (80px avg)":
            if contour is not None:
                pts = contour.reshape(-1, 2)

                # get nearest point index
                dists = np.sqrt((pts[:, 0] - gx)**2 + (pts[:, 1] - gy)**2)
                idx = np.argmin(dists)

                # grab a segment ~80px
                n_pts = min(40, len(pts) // 4)
                half = n_pts // 2

                s_idx = (idx - half) % len(pts)
                e_idx = (idx + half) % len(pts)

                if s_idx < e_idx:
                    seg = pts[s_idx:e_idx+1]
                else:
                    seg = np.vstack([pts[s_idx:], pts[:e_idx+1]])

                if len(seg) > 2:
                    # PCA approach for tangent
                    xs = seg[:, 0]
                    ys = seg[:, 1]
                    mx, my = np.mean(xs), np.mean(ys)

                    x_c = xs - mx
                    y_c = ys - my

                    Sxx = np.sum(x_c * x_c)
                    Sxy = np.sum(x_c * y_c)
                    Syy = np.sum(y_c * y_c)

                    if Sxx + Syy > 1e-6:
                        tr = Sxx + Syy
                        det = Sxx * Syy - Sxy * Sxy
                        l1 = tr/2 + np.sqrt(max(0, (tr/2)**2 - det))

                        if abs(Sxy) > 1e-6:
                            tx = l1 - Syy
                            ty = Sxy
                        elif abs(Sxx - l1) > 1e-6:
                            tx = Sxy
                            ty = l1 - Sxx
                        else:
                            tx = seg[-1, 0] - seg[0, 0]
                            ty = seg[-1, 1] - seg[0, 1]

                        l = np.sqrt(tx**2 + ty**2)
                        if l > 1e-6:
                            tx /= l; ty /= l
                            # rotate 90 deg for normal
                            dx = -ty
                            dy = tx
                        else:
                            dx, dy = 1.0, 0.0
                    else:
                        dx, dy = 1.0, 0.0
                else:
                    if len(seg) >= 2:
                        tx = seg[-1, 0] - seg[0, 0]
                        ty = seg[-1, 1] - seg[0, 1]
                        l = np.sqrt(tx**2 + ty**2)
                        if l > 1e-6:
                            tx /= l; ty /= l
                            dx, dy = -ty, tx

        elif mode == "Depth Gradients":
            if 0 <= gy < vec_y.shape[0] and 0 <= gx < vec_x.shape[1]:
                val_x = vec_x[gy, gx]
                val_y = vec_y[gy, gx]
            else:
                ks = 3
                val_x = np.mean(vec_x[max(0, gy-ks):min(h, gy+ks+1), max(0, gx-ks):min(w, gx+ks+1)])
                val_y = np.mean(vec_y[max(0, gy-ks):min(h, gy+ks+1), max(0, gx-ks):min(w, gx+ks+1)])

            lenght = np.sqrt(val_x**2 + val_y**2)
            if lenght < 1e-6:
                # radial fallback
                dx = gx - w // 2
                dy = gy - h // 2
                lenght = np.sqrt(dx**2 + dy**2)
                if lenght > 1e-6: dx /= lenght; dy /= lenght
            else:
                # perpendicular
                dx = -val_y / lenght
                dy = val_x / lenght

        elif mode == "Image Edges":
             ks = 5
             ymin, ymax = max(0, gy - ks), min(h, gy + ks + 1)
             xmin, xmax = max(0, gx - ks), min(w, gx + ks + 1)

             local = mask[ymin:ymax, xmin:xmax].astype(np.float32)
             if local.size > 0:
                 lx = cv2.Sobel(local, cv2.CV_64F, 1, 0, ksize=3)
                 ly = cv2.Sobel(local, cv2.CV_64F, 0, 1, ksize=3)

                 cy = min(ks, gy - ymin)
                 cx = min(ks, gx - xmin)

                 if cy < ly.shape[0] and cx < lx.shape[1]:
                     vx, vy = lx[cy, cx], ly[cy, cx]
                 else:
                     vx, vy = np.mean(lx), np.mean(ly)

                 l = np.sqrt(vx**2 + vy**2)
                 if l > 1e-6:
                     dx = -vy / l
                     dy = vx / l

        elif mode == "Radial from Center":
            cx, cy = w//2, h//2
            dx = gx - cx
            dy = gy - cy
            l = np.sqrt(dx**2 + dy**2)
            if l > 1e-6:
                dx /= l
                dy /= l

        # Trace both ways
        x1, y1 = self._trace_line(mask, gx, gy, dx, dy, limit)
        x2, y2 = self._trace_line(mask, gx, gy, -dx, -dy, limit)

        full_dist = np.sqrt((x1-x2)**2 + (y1-y2)**2)
        return x1, y1, x2, y2, full_dist

    def _cog_ray(self, mask, gp, cog, limit=500):
        gx, gy = gp
        cx, cy = cog

        vx = gx - cx
        vy = gy - cy

        dist = np.sqrt(vx**2 + vy**2)
        if dist < 1e-6: return cx, cy, cx, cy, 0.0

        vx /= dist
        vy /= dist

        # Forward and back
        e1x, e1y = self._trace_line(mask, cx, cy, vx, vy, limit)
        e2x, e2y = self._trace_line(mask, cx, cy, -vx, -vy, limit)

        l = np.sqrt((e1x-e2x)**2 + (e1y-e2y)**2)
        return e1x, e1y, e2x, e2y, l

    def _trace_line(self, mask, sx, sy, dx, dy, limit):
        h, w = mask.shape
        cx, cy = float(sx), float(sy)

        GAP_MAX = 10
        gap = 0
        last_x, last_y = int(cx), int(cy)

        for _ in range(limit):
            cx += dx
            cy += dy
            ix, iy = int(round(cx)), int(round(cy))

            if ix < 0 or ix >= w or iy < 0 or iy >= h:
                return last_x, last_y

            if mask[iy, ix] == 0:
                gap += 1
                if gap > GAP_MAX:
                    return last_x, last_y
            else:
                gap = 0
                last_x, last_y = ix, iy

        return last_x, last_y

    def get_mask_data(self, img, dmap, pct=60):
        # 1. Depth threshold
        thresh = np.percentile(dmap, pct)
        droi = (dmap >= thresh).astype(np.uint8) * 255

        k1 = np.ones((7,7), np.uint8)
        droi = cv2.morphologyEx(droi, cv2.MORPH_CLOSE, k1)
        droi = cv2.morphologyEx(droi, cv2.MORPH_OPEN, k1)

        # 2. Canny
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 30, 100)

        # 3. Combine
        e_roi = cv2.bitwise_and(edges, edges, mask=droi)
        dilated = cv2.dilate(e_roi, np.ones((5,5), np.uint8), iterations=3)

        blend = (droi.astype(np.float32) * 0.2 + dilated.astype(np.float32) * 0.8).astype(np.uint8)
        _, bin_mask = cv2.threshold(blend, 100, 255, cv2.THRESH_BINARY)

        # 4. Clean
        final_mask = cv2.morphologyEx(bin_mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))
        final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))

        # 5. Contours
        conts, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        main_c = None

        if conts:
            area_tot = final_mask.shape[0] * final_mask.shape[1]
            valid = [c for c in conts if 0.005 * area_tot < cv2.contourArea(c) < 0.40 * area_tot]
            if valid:
                main_c = max(valid, key=cv2.contourArea)
            else:
                main_c = max(conts, key=cv2.contourArea)

            final_mask = np.zeros_like(final_mask)
            cv2.drawContours(final_mask, [main_c], -1, 255, cv2.FILLED)

        # 6. Grads
        d8 = (dmap * 255).astype(np.uint8)
        gx = cv2.Sobel(d8, cv2.CV_64F, 1, 0, ksize=5)
        gy = cv2.Sobel(d8, cv2.CV_64F, 0, 1, ksize=5)

        return final_mask, main_c, gx, gy, e_roi

    def process(self, rgb, depth, n_grasps=10, pct=60, mult=3, min_l=100, max_l=1000,
                algo="Through CoG", boost=0.0, grad_src="Depth Gradients"):

        h, w = rgb.shape[:2]
        info = {}

        mask, cont, gx, gy, ed = self.get_mask_data(rgb, depth, pct)

        # normalize depth grad
        mag = np.sqrt(gx**2 + gy**2)
        norm_g = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)

        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        enorm = cv2.Canny(gray, 50, 150).astype(np.float32) / 255.0

        # COG
        M = cv2.moments(mask)
        if M["m00"] != 0:
            cx, cy = int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])
        else:
            cx, cy = w//2, h//2

        info['mask_area'] = np.count_nonzero(mask)
        info['mask_valid'] = info['mask_area'] > 0
        info['edges'] = ed
        info['depth_grad'] = norm_g
        info['mask'] = mask
        info['cog'] = (cx, cy)

        # Sample points
        if cont is not None:
            pts = cont.reshape(-1, 2)
            # take 100 points roughly
            idx = np.linspace(0, len(pts)-1, min(len(pts), 100), dtype=int)
            cands = pts[idx]
        else:
            cands = [[w//2, h//2]]

        info['num_contour_points'] = len(cands)

        # Rank candidates
        prelim = []
        diag = np.sqrt(w**2 + h**2)

        for p in cands:
            px, py = p
            if not (0 <= py < h and 0 <= px < w): continue

            eq = enorm[py, px]
            dq = norm_g[py, px]
            dist = np.sqrt((px - cx)**2 + (py - cy)**2)
            cq = 1 - (dist / diag)

            score = (self.we * eq + self.wd * dq + self.wc * cq)

            prelim.append({
                'p': (px, py),
                'score': score,
                'eq': eq, 'dq': dq, 'cq': cq
            })

        info['num_preliminary'] = len(prelim)
        prelim.sort(key=lambda x: x['score'], reverse=True)

        top = prelim[:n_grasps * mult]
        info['num_evaluated'] = len(top)

        final_grasps = []
        rejects = {'too_short': 0, 'too_long': 0, 'zero_length': 0}

        # Ray cast logic
        for t in top:
            px, py = t['p']

            if algo == "Through CoG":
                ex1, ey1, ex2, ey2, length = self._cog_ray(mask, (px, py), (cx, cy))
            else:
                ex1, ey1, ex2, ey2, length = self._ray_cast(mask, (px, py), gx, gy, cont, grad_src)

            if length <= min_l:
                if length == 0: rejects['zero_length'] += 1
                else: rejects['too_short'] += 1
                continue
            if length >= max_l:
                rejects['too_long'] += 1
                continue

            dx, dy = ex1 - ex2, ey1 - ey2
            ang = np.arctan2(dy, dx)
            mx, my = (ex1 + ex2)/2, (ey1 + ey2)/2

            rank_score = length
            if algo == "Direct Line with CoG Boost":
                 d2cog = np.sqrt((mx - cx)**2 + (my - cy)**2)
                 prox = 1 - (d2cog / diag)
                 rank_score = length - (boost * prox * 500)

            final_grasps.append(Candidate(
                x=mx, y=my, angle=ang, w=length,
                eq=t['eq'], dq=t['dq'], cq=t['cq'], comb=t['score'],
                l=rank_score
            ))

        final_grasps.sort(key=lambda x: x.line_length)

        info['num_valid'] = len(final_grasps)
        info['invalid_grasps'] = rejects
        info['has_contour'] = cont is not None
        # remap keys for debug viz
        viz_cands = [{'point': x['p'], 'combined_quality': x['score']} for x in top]
        info['top_candidates'] = viz_cands[:20] if len(viz_cands) >= 20 else viz_cands

        return final_grasps[:n_grasps], info

class AppWorkingGraspDetector:

    def __init__(self, edge_weight=0.20, depth_weight=0.40, cog_weight=0.40):
        self.detector = GraspDetector(edge_weight, depth_weight, cog_weight)

    def detect(self, rgb: np.ndarray, depth: np.ndarray, num_grasps: int = 10):
        grasps, info = self.detector.process(
            rgb=rgb,
            depth=depth,
            n_grasps=num_grasps,
            pct=DEPTH_PERCENTILE,
            mult=CANDIDATE_MULTIPLIER,
            min_l=MIN_GRASP_LENGTH,
            max_l=MAX_GRASP_LENGTH,
            algo=RAY_ALGORITHM,
            boost=COG_BOOST_VALUE,
            grad_src=GRADIENT_SOURCE
        )

        return grasps, info


PIPELINE PARAMETERS (app_working.py method)
Weights: Edge=0.00, Depth=0.00, CoG=1.00
Depth Percentile: 30
Ray Algorithm: Direct Line with CoG Boost
Grasp Length Range: [1, 1000]
Output Grasps: 1, Candidate Multiplier: 100


## 7. Evaluation Metrics


In [ ]:
class GraspEvaluator:

    def __init__(self, iou_thresh: float = 0.25, angle_thresh: float = 30.0):
        self.iou_thresh = iou_thresh
        self.angle_thresh = angle_thresh

    def polygon_iou(self, poly1: np.ndarray, poly2: np.ndarray):
        """Calculate IoU between two polygons using mask-based approach."""
        # Get bounding box to determine mask size
        all_pts = np.vstack([poly1, poly2])
        min_x, min_y = np.floor(all_pts.min(axis=0)).astype(int)
        max_x, max_y = np.ceil(all_pts.max(axis=0)).astype(int)
        
        # Add padding
        min_x, min_y = max(0, min_x - 1), max(0, min_y - 1)
        max_x, max_y = max_x + 1, max_y + 1
        
        w = max_x - min_x
        h = max_y - min_y
        
        if w <= 0 or h <= 0:
            return 0.0
        
        # Create masks
        mask1 = np.zeros((h, w), dtype=np.uint8)
        mask2 = np.zeros((h, w), dtype=np.uint8)
        
        # Translate polygons to mask coordinates
        pts1 = np.clip((poly1 - [min_x, min_y]).astype(np.int32), 0, [w-1, h-1])
        pts2 = np.clip((poly2 - [min_x, min_y]).astype(np.int32), 0, [w-1, h-1])
        
        cv2.fillPoly(mask1, [pts1], 1)
        cv2.fillPoly(mask2, [pts2], 1)
        
        intersection = np.sum(mask1 & mask2)
        union = np.sum(mask1 | mask2)
        
        return intersection / max(union, 1e-8)

    def angle_diff(self, angle1: float, angle2: float):
        """Calculate minimum angle difference in degrees."""
        diff = abs(angle1 - angle2)
        diff = min(diff, np.pi - diff, abs(diff - np.pi))
        
        return np.degrees(diff)

    def evaluate_grasp(self, prediction: Candidate, ground_truths: List[Dict]):
        """Evaluate a single predicted grasp against all ground truths."""
        pred_corners = prediction.get_corners()
        
        best_iou = 0.0
        best_angle_diff = 180.0
        
        for gt in ground_truths:
            iou = self.polygon_iou(pred_corners, gt['corners'])
            angle_diff = self.angle_diff(prediction.angle, gt['angle'])
            
            if iou > best_iou:
                best_iou = iou
                best_angle_diff = angle_diff
        
        success = (best_iou >= self.iou_thresh and
                   best_angle_diff <= self.angle_thresh)
        
        return {'success': success, 'best_iou': best_iou, 'best_angle_diff': best_angle_diff}

    def evaluate_image(self, predictions: List[Candidate], ground_truths: List[Dict]):
        """Evaluate all predictions for a single image."""
        if not predictions:
            return {'top1_success': False, 'top5_success': False, 'avg_iou': 0.0}
        
        results = [self.evaluate_grasp(pred, ground_truths) for pred in predictions]
        
        return {
            'top1_success': results[0]['success'] if results else False,
            'top5_success': any(r['success'] for r in results[:5]),
            'any_success': any(r['success'] for r in results),
            'avg_iou': np.mean([r['best_iou'] for r in results]),
            'avg_angle_diff': np.mean([r['best_angle_diff'] for r in results]),
        }

## 8. Run Full Evaluation


In [22]:
def run_evaluation(samples, depth_model, detector, evaluator, num_grasps=10, visualize_every=10):
    results = {
        'top1_successes': 0, 'top5_successes': 0, 'any_successes': 0,
        'total_samples': 0, 'ious': [], 'angle_diffs': [],
        'depth_times': [], 'detection_times': [], 'total_times': [],
        'visualization_samples': []
    }

    print("\n" + "="*80)
    print("RUNNING EVALUATION")
    print("="*80)

    for i, sample in enumerate(tqdm(samples, desc="Processing")):
        try:
            rgb = cv2.imread(sample['rgb_path'])
            if rgb is None:
                continue
            rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

            start_total = time.time()

            # Depth estimation
            depth, depth_time = depth_model.predict(rgb)
            results['depth_times'].append(depth_time)

            # Grasp detection
            start_detect = time.time()
            predictions, debug = detector.detect(rgb, depth, num_grasps)
            detection_time = (time.time() - start_detect) * 1000
            results['detection_times'].append(detection_time)

            total_time = (time.time() - start_total) * 1000
            results['total_times'].append(total_time)

            # Evaluation
            eval_result = evaluator.evaluate_image(predictions, sample['grasps'])

            results['total_samples'] += 1
            if eval_result['top1_success']:
                results['top1_successes'] += 1
            if eval_result['top5_success']:
                results['top5_successes'] += 1
            if eval_result['any_success']:
                results['any_successes'] += 1

            results['ious'].append(eval_result['avg_iou'])
            results['angle_diffs'].append(eval_result['avg_angle_diff'])

            if i % visualize_every == 0:
                results['visualization_samples'].append({
                    'rgb': rgb, 'depth': depth, 'predictions': predictions,
                    'ground_truths': sample['grasps'], 'debug': debug,
                    'eval': eval_result, 'sample_id': sample['id']
                })

        except Exception as e:
            print(f"\nError: {e}")
            continue

    # Calculate metrics
    n = results['total_samples']
    if n > 0:
        results['top1_accuracy'] = results['top1_successes'] / n * 100
        results['top5_accuracy'] = results['top5_successes'] / n * 100
        results['any_accuracy'] = results['any_successes'] / n * 100
        results['mean_iou'] = np.mean(results['ious'])
        results['mean_angle_diff'] = np.mean(results['angle_diffs'])
        results['mean_depth_time'] = np.mean(results['depth_times'])
        results['mean_detection_time'] = np.mean(results['detection_times'])
        results['mean_total_time'] = np.mean(results['total_times'])
        results['std_total_time'] = np.std(results['total_times'])

    return results


In [23]:

detector = AppWorkingGraspDetector(W_EDGE, W_DEPTH, W_COG)

evaluator = GraspEvaluator(IOU_THRESHOLD, ANGLE_THRESHOLD)

if samples:
    eval_results = run_evaluation(samples, depth_model, detector, evaluator,
                                   num_grasps=NUM_OUTPUT_GRASPS, visualize_every=5)
else:
    print("No samples to evaluate. Please download the dataset first.")
    eval_results = {}



RUNNING EVALUATION


Processing:   0%|          | 1/850 [00:00<13:02,  1.09it/s]


Error: 'GraspEvaluator' object has no attribute 'evaluate_image'


Processing:   0%|          | 2/850 [00:01<09:42,  1.46it/s]


Error: 'GraspEvaluator' object has no attribute 'evaluate_image'


...

Processing:   1%|          | 8/850 [00:04<07:19,  1.92it/s]


Error: 'GraspEvaluator' object has no attribute 'evaluate_image'


Processing:   1%|          | 9/850 [00:05<08:01,  1.75it/s]


Error: 'GraspEvaluator' object has no attribute 'evaluate_image'


KeyboardInterrupt: 

In [ ]:
def print_results(results):
    print("\n" + "="*80)
    print("EVALUATION RESULTS: DepthAnythingV2 Small + Heuristic Grasp Detection")
    print("="*80)

    print("\n ACCURACY METRICS")
    print("-"*40)
    print(f"  Total Samples:        {results.get('total_samples', 0)}")
    print(f"  Top-1 Accuracy:       {results.get('top1_accuracy', 0):.2f}%")
    print(f"  Top-5 Accuracy:       {results.get('top5_accuracy', 0):.2f}%")
    print(f"  Mean IoU:             {results.get('mean_iou', 0):.4f}")
    print(f"  Mean Angle Diff:      {results.get('mean_angle_diff', 0):.2f}°")

    print("\n⏱ TIMING METRICS")
    print("-"*40)
    print(f"  Depth Estimation:     {results.get('mean_depth_time', 0):.2f} ms")
    print(f"  Grasp Detection:      {results.get('mean_detection_time', 0):.2f} ms")
    print(f"  Total Pipeline:       {results.get('mean_total_time', 0):.2f} ms")
    print(f"  Throughput:           {1000/max(results.get('mean_total_time', 1), 1):.2f} FPS")
    print("="*80)

if eval_results:
    print_results(eval_results)


EVALUATION RESULTS: DepthAnythingV2 Small + Heuristic Grasp Detection

📊 ACCURACY METRICS
----------------------------------------
  Total Samples:        800
  Top-1 Accuracy:       71.75%
  Top-5 Accuracy:       71.75%
  Any Success Rate:     71.75%
  Mean IoU:             0.3933
  Mean Angle Diff:      21.32°

⏱️  TIMING METRICS
----------------------------------------
  Depth Estimation:     404.67 ms
  Grasp Detection:      26.96 ms
  Total Pipeline:       432.39 ms
  Throughput:           2.31 FPS


## 10. Visualization


In [ ]:
def visualize_sample(sample_data, figsize=(16, 10)):
    rgb = sample_data['rgb']
    depth = sample_data['depth']
    predictions = sample_data['predictions']
    ground_truths = sample_data['ground_truths']
    debug = sample_data['debug']
    eval_result = sample_data['eval']

    fig, axes = plt.subplots(2, 3, figsize=figsize)

    # ROW 1: Input Data

    # 1. RGB Input
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('RGB Input', fontweight='bold', fontsize=12)
    axes[0, 0].axis('off')

    # 2. Depth Map
    axes[0, 1].imshow(depth, cmap='magma')
    axes[0, 1].set_title('DepthAnythingV2 Small', fontweight='bold', fontsize=12)
    axes[0, 1].axis('off')

    # 3. OBJECT MASK
    mask_percentage = 0
    if 'mask' in debug and debug['mask'] is not None:
        axes[0, 2].imshow(debug['mask'], cmap='gray')
        axes[0, 2].set_title('Object Mask', fontweight='bold', fontsize=12, color='green')

        # Overlay CoG on mask
        cog = debug.get('cog', None)
        if cog is not None:
            axes[0, 2].scatter([cog[0]], [cog[1]], c='cyan', s=300, marker='*',
                              edgecolors='white', linewidths=2)
            axes[0, 2].text(cog[0] + 20, cog[1], 'CoG', color='cyan', fontsize=11,
                           fontweight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

        # Add mask statistics
        mask_area = np.sum(debug['mask'] > 0)
        total_area = debug['mask'].shape[0] * debug['mask'].shape[1]
        mask_percentage = (mask_area / total_area) * 100
        axes[0, 2].text(10, 30, f'Coverage: {mask_percentage:.1f}%',
                       color='white', fontsize=10, fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    else:
        if 'saliency' in debug:
            axes[0, 2].imshow(debug['saliency'], cmap='hot')
            axes[0, 2].set_title('Saliency Map (Mask N/A)', fontweight='bold', fontsize=12, color='orange')
        else:
            axes[0, 2].imshow(rgb)
            axes[0, 2].set_title('Mask Not Available', fontweight='bold', fontsize=12, color='red')
    axes[0, 2].axis('off')

    # ROW 2: Results

    # 4. Ground Truth
    axes[1, 0].imshow(rgb)
    for gt in ground_truths[:5]:
        polygon = Polygon(gt['corners'], fill=False, edgecolor='blue', linewidth=2, linestyle='--')
        axes[1, 0].add_patch(polygon)
    axes[1, 0].set_title(f'Ground Truth ({len(ground_truths)} grasps)', fontweight='bold', fontsize=12)
    axes[1, 0].axis('off')

    # 5. Top 5 Predictions
    axes[1, 1].imshow(rgb)
    cmap = plt.cm.get_cmap('RdYlGn_r')
    for i, pred in enumerate(predictions[:5]):
        corners = pred.get_corners()
        color = cmap(i / max(len(predictions[:5]) - 1, 1))
        polygon = Polygon(corners, fill=False, edgecolor=color, linewidth=2)
        axes[1, 1].add_patch(polygon)

        axes[1, 1].text(pred.x + 10, pred.y - 10, f'#{i+1}',
                       color='white', fontsize=9, fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))

    # Draw CoG
    cog = debug.get('cog', (320, 240))
    axes[1, 1].scatter([cog[0]], [cog[1]], c='cyan', s=200, marker='*', edgecolors='black', linewidths=2)
    axes[1, 1].set_title('Top 5 Predictions', fontweight='bold', fontsize=12)
    axes[1, 1].axis('off')

    # 6. Success/Fail Comparison
    axes[1, 2].imshow(rgb)

    # Draw ground truth
    for gt in ground_truths[:2]:
        polygon = Polygon(gt['corners'], fill=False, edgecolor='blue', linewidth=2, linestyle='--')
        axes[1, 2].add_patch(polygon)

    # Draw best prediction
    if predictions:
        pred = predictions[0]
        color = 'green' if eval_result['top1_success'] else 'red'
        polygon = Polygon(pred.get_corners(), fill=False, edgecolor=color, linewidth=3)
        axes[1, 2].add_patch(polygon)

    # Status text
    status = '✓ PASS' if eval_result['top1_success'] else '✗ FAIL'
    iou = eval_result.get('best_iou', eval_result.get('avg_iou', 0))
    axes[1, 2].set_title(f'{status} (IoU: {iou:.3f})', fontweight='bold', fontsize=12,
                         color='green' if eval_result['top1_success'] else 'red')
    axes[1, 2].axis('off')

    # Overall title
    plt.suptitle(f"Sample: {sample_data['sample_id']}", fontsize=14, fontweight='bold')
    plt.tight_layout()

    # PRINT DIAGNOSTIC INFO
    print(f"\n{'='*80}")
    print(f"Sample: {sample_data['sample_id']}")
    print(f"{'='*80}")
    print(f"Status: {status}")
    print(f"IoU: {iou:.4f}")

    # Mask diagnostics
    if 'mask' in debug and debug['mask'] is not None:
        mask = debug['mask']
        mask_pixels = np.sum(mask > 0)
        total_pixels = mask.shape[0] * mask.shape[1]
        print(f"\nMask Diagnostics:")
        print(f"  Mask coverage: {mask_pixels}/{total_pixels} pixels ({mask_percentage:.1f}%)")
        print(f"  Mask size: {mask.shape}")

        # Check if mask is reasonable
        if mask_percentage < 5:
            print(f"  WARNING: Mask is very small (<5%)")
        elif mask_percentage > 50:
            print(f"  WARNING: Mask is very large (>50%)")
        else:
            print(f"  Mask size looks reasonable")
    else:
        print(f"\nNo mask available in debug info")

    # CoG diagnostics
    if cog is not None:
        print(f"\nCenter of Gravity: ({cog[0]}, {cog[1]})")

    # Prediction diagnostics
    if predictions:
        print(f"\nTop Prediction Quality:")
        print(f"  Edge Quality:  {predictions[0].edge_quality:.3f}")
        print(f"  Depth Quality: {predictions[0].depth_quality:.3f}")
        print(f"  CoG Quality:   {predictions[0].cog_quality:.3f}")
        print(f"  Combined:      {predictions[0].combined_quality:.3f}")
        print(f"  Position:      ({predictions[0].x}, {predictions[0].y})")

    print(f"{'='*80}\n")

    return fig

# Visualize samples
vis_samples = eval_results.get('visualization_samples', [])
if vis_samples:
    print(f"\n🔍 VISUALIZING {min(100, len(vis_samples))} SAMPLES FOR DEBUGGING\n")
    for i, sample_data in enumerate(vis_samples[:100]):
        print(f"\n{'#'*80}")
        print(f"# VISUALIZATION {i+1}/{min(100, len(vis_samples))}")
        print(f"{'#'*80}")
        fig = visualize_sample(sample_data)
        plt.show()
else:
    print("No visualization samples available.")

In [ ]:
import os
import matplotlib.pyplot as plt

output_dir = "grasp_visualizations"
os.makedirs(output_dir, exist_ok=True)

vis_samples = eval_results.get('visualization_samples', [])

if vis_samples:
    print(f"\n🔍 PROCESSING {len(vis_samples)} SAMPLES...")
    print(f"📁 Images will be saved to: {output_dir}")

    for i, sample_data in enumerate(vis_samples):
        fig = visualize_sample(sample_data)

        # Save to file
        save_path = os.path.join(output_dir, f"sample_{i:03d}_{sample_data['sample_id']}.png")
        plt.savefig(save_path, bbox_inches='tight', dpi=100)

        plt.close(fig)

        if (i + 1) % 50 == 0:
            print(f"Saved {i + 1}/{len(vis_samples)} images...")

    print(f"\n✅ Done! All images saved in '{output_dir}'")

    import shutil
    shutil.make_archive('grasp_visualizations', 'zip', output_dir)
    print("Prepared 'grasp_visualizations.zip' for download.")

else:
    print("No visualization samples available.")

## Grid search for optimal params



In [ ]:
def optimize_appworking_with_depth_cache(
    samples,
    depth_model,
    evaluator,
    grid_search_samples=5,
    edge_range=(0.0, 0.1),
    depth_range=(0.0, 0.1),
    cog_range=(0.9, 1.0),
    weight_step=0.05,
    depth_percentiles=(55, 60, 65),
    ray_algorithms=("Direct Line with CoG Boost",),
    gradient_sources=("Contour Direction (80px avg)",),
    cog_boosts=(2.0, 2.5, 1.5, 1.0, 3.0),
    min_grasp_lengths=(1, 2, 5, 10),
    max_grasp_lengths=(1000,),
    candidate_multipliers=(10,),
    num_grasps=10,
):
    print("=" * 80)
    print("STEP 1: Pre computing depth maps for AppWorking grid search")
    print("=" * 80)

    if grid_search_samples and len(samples) > grid_search_samples:
        np.random.seed(42)
        indices = np.random.permutation(len(samples))[:grid_search_samples]
        eval_samples = [samples[i] for i in indices]
    else:
        eval_samples = samples

    cached_data = []
    for sample in tqdm(eval_samples, desc="Computing depth maps"):
        try:
            rgb = cv2.imread(sample["rgb_path"])
            if rgb is None:
                continue
            rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
            depth, _ = depth_model.predict(rgb)
            cached_data.append({
                "rgb": rgb,
                "depth": depth,
                "grasps": sample["grasps"],
                "id": sample.get("id", None),
            })
        except Exception:
            continue

    if not cached_data:
        raise RuntimeError("No valid samples cached for grid search.")

    weight_combos = generate_weight_combinations(
        edge_range=edge_range,
        depth_range=depth_range,
        cog_range=cog_range,
        step=weight_step,
    )

    param_combinations = []
    for (w_edge, w_depth, w_cog) in weight_combos:
        for depth_pct in depth_percentiles:
            for ray_algo in ray_algorithms:
                for grad_src in gradient_sources:
                    for boost in cog_boosts:
                        for min_len in min_grasp_lengths:
                            for max_len in max_grasp_lengths:
                                for mult in candidate_multipliers:
                                    param_combinations.append({
                                        "w_edge": w_edge,
                                        "w_depth": w_depth,
                                        "w_cog": w_cog,
                                        "depth_percentile": depth_pct,
                                        "ray_algorithm": ray_algo,
                                        "gradient_source": grad_src,
                                        "cog_boost": boost,
                                        "min_grasp_length": min_len,
                                        "max_grasp_length": max_len,
                                        "candidate_multiplier": mult,
                                    })

    print("\n" + "=" * 80)
    print(f"STEP 2: Grid search over {len(param_combinations)} parameter combinations")
    print(f"Each combination tests {len(cached_data)} cached samples")
    print("=" * 80)

    # STEP 3: run grid search
    all_results = []
    best_top1 = 0.0
    best_result = None

    start_time = time.time()

    for i, params in enumerate(tqdm(param_combinations, desc="Grid search")):
        detector = AppWorkingGraspDetectorEnhanced(
            edge_weight=params["w_edge"],
            depth_weight=params["w_depth"],
            cog_weight=params["w_cog"],
            depth_percentile=params["depth_percentile"],
            candidate_multiplier=params["candidate_multiplier"],
            min_grasp_length=params["min_grasp_length"],
            max_grasp_length=params["max_grasp_length"],
            ray_algorithm=params["ray_algorithm"],
            cog_boost=params["cog_boost"],
            gradient_source=params["gradient_source"],
        )

        stats = {
            "top1": 0,
            "top5": 0,
            "any": 0,
            "total": 0,
            "ious": [],
            "angles": [],
        }

        for item in cached_data:
          try:
              grasps, _ = detector.detect(item["rgb"], item["depth"], num_grasps=num_grasps)

              eval_r = evaluator.evaluate_image(grasps, item["grasps"])

              stats["total"] += 1
              if eval_r["top1_success"]:
                  stats["top1"] += 1
              if eval_r["top5_success"]:
                  stats["top5"] += 1
              if eval_r.get("any_success", False):
                  stats["any"] += 1
              stats["ious"].append(eval_r["avg_iou"])
              stats["angles"].append(eval_r["avg_angle_diff"])
          except Exception:
            continue

        if stats["total"] > 0:
            top1_acc = stats["top1"] / stats["total"] * 100.0
            top5_acc = stats["top5"] / stats["total"] * 100.0
            any_acc = stats["any"] / stats["total"] * 100.0
            avg_iou = float(np.mean(stats["ious"])) if stats["ious"] else 0.0
            avg_angle = float(np.mean(stats["angles"])) if stats["angles"] else 0.0
        else:
            top1_acc = top5_acc = any_acc = avg_iou = avg_angle = 0.0

        result_row = {
            "edge_weight": params["w_edge"],
            "depth_weight": params["w_depth"],
            "cog_weight": params["w_cog"],
            "depth_percentile": params["depth_percentile"],
            "ray_algorithm": params["ray_algorithm"],
            "gradient_source": params["gradient_source"],
            "cog_boost": params["cog_boost"],
            "min_grasp_length": params["min_grasp_length"],
            "max_grasp_length": params["max_grasp_length"],
            "candidate_multiplier": params["candidate_multiplier"],
            "top1_accuracy": top1_acc,
            "top5_accuracy": top5_acc,
            "any_accuracy": any_acc,
            "avg_iou": avg_iou,
            "avg_angle_diff": avg_angle,
        }
        all_results.append(result_row)

        if top1_acc > best_top1:
            best_top1 = top1_acc
            best_result = result_row

        # occasional progress update
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            print(
                f"Checked {i+1}/{len(param_combinations)} combos | "
                f"Current best Top1={best_top1:.1f}% | "
                f"Elapsed={elapsed:.0f}s"
            )

    elapsed_total = time.time() - start_time
    df = pd.DataFrame(all_results)

    print("\n" + "=" * 80)
    print("=" * 80)
    print(f"Time: {elapsed_total:.1f}s")
    if best_result is not None:
        print("\nBest parameter set:")
        for k, v in best_result.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")

    return {
        "best_params": best_result,
        "best_accuracy": best_top1,
        "all_results": df,
        "elapsed_time": elapsed_total,
    }


In [ ]:
optimization_results = optimize_appworking_with_depth_cache(
    samples=samples,
    depth_model=depth_model,
    evaluator=evaluator,
    grid_search_samples=830,

    edge_range=(0.0, 0.1),
    depth_range=(0.0, 0.2),
    cog_range=(0.7, 1.0),
    weight_step=0.1,

    # Discrete option sets:
    depth_percentiles=[22.5, 27.5, 30, 35],
    ray_algorithms=["Direct Line with CoG Boost"],
    gradient_sources=["Contour Direction (80px avg)"],
    cog_boosts=[3.25, 3.75, 4.25,],
    min_grasp_lengths=[1],
    max_grasp_lengths=[1000],
    candidate_multipliers=[30],

    num_grasps=10,
)

df = optimization_results["all_results"]
df.to_csv("appworking_optimization_full.csv", index=False)
print("Saved results to appworking_optimization_full.csv")


STEP 1: Pre computing depth maps for AppWorking grid search


Computing depth maps: 100%|██████████| 800/800 [05:14<00:00,  2.54it/s]


✓ Cached 800 samples with depth maps

STEP 2: Grid search over 84 parameter combinations
Each combination tests 800 cached samples


Grid search:   2%|▏         | 2/84 [02:13<1:31:28, 66.93s/it]


KeyboardInterrupt: 